# CUDA Variant 3: Grid-Stride Loop with Prefetch (no page creation, no mem advise)

**Kernel:** Vector AXPY, `Z[i] = a * X[i] + Y[i]`

| Item | Value |
|---|---|
| Initialization | `X[i] = sinf(i * 0.001f)`, `Y[i] = cosf(i * 0.002f)`, `Z[i]` untouched on host, `a = 2.0f` |
| Vector sizes | 1003 (boundary), 2^20, 2^20 + 7 (boundary), 2^26, 2^28 |
| Block sizes | 256, 512, 1024 |
| Runs per kernel | 30 (avg / min / max, plus 1st run) |
| Answer key | C version (`axpy_c`), tolerance 1e-5 (GPU may fuse multiply-add into FMA) |

**Variant definition used here**
* Grid-stride loop kernel.
* `cudaMemPrefetchAsync` on the **input** arrays `X` and `Y` only.
* *No page creation*: the output array `Z` is not prefetched, so its pages are first created on the GPU when the kernel touches them (visible as a slower 1st run).
* *No mem advise*: no `cudaMemAdvise` calls.

Runtime: Colab with a GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi
!nvcc --version

Mon Nov  9 14:10:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Variant 3: CUDA grid-stride loop, with prefetch, without page creation, without mem advise

In [ ]:
%%writefile cuda_prefetch.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define NUM_RUNS 30   // minimum 30 runs per kernel

// ---------------------------------------------------------------
// GPU kernel: Z = a * X + Y  (grid-stride loop)
// ---------------------------------------------------------------
__global__
void axpy(int n, float a, float* d_X, float* d_Y, float* d_Z)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = index; i < n; i += stride)
        d_Z[i] = a * d_X[i] + d_Y[i];
}

// ---------------------------------------------------------------
// C kernel: sanity check answer key
// ---------------------------------------------------------------
void axpy_c(int n, float a, float* h_X, float* h_Y, float* h_out)
{
    for (int i = 0; i < n; i++)
        h_out[i] = a * h_X[i] + h_Y[i];
}

// Correctness check (tolerance because the GPU may fuse a*X+Y into one FMA)
int compare(float* ref, float* target, int size)
{
    int ctr = 0;
    for (int i = 0; i < size; i++) {
        float diff = fabsf(ref[i] - target[i]);
        float tol = 1e-5f * fmaxf(1.0f, fabsf(ref[i]));
        if (diff > tol) ctr++;
    }
    return ctr;
}

void print_samples(const char* kernel_name, float* Z, int n)
{
    printf("%s Elements:\n  First 5: %.4f %.4f %.4f %.4f %.4f\n",
           kernel_name, Z[0], Z[1], Z[2], Z[3], Z[4]);
    printf("  Last 5:  %.4f %.4f %.4f %.4f %.4f\n",
           Z[n-5], Z[n-4], Z[n-3], Z[n-2], Z[n-1]);
}

void run_test(int ARRAY_SIZE, float a, int numThreads)
{
    printf("\n======================================================================\n");
    printf(" CUDA (grid-stride + prefetch, NO page creation, NO mem advise)\n");
    printf(" n = %d, a = %.2f, block size = %d\n", ARRAY_SIZE, a, numThreads);
    printf("======================================================================\n");

    const size_t ARRAY_BYTES = (size_t)ARRAY_SIZE * sizeof(float);

    // declare arrays (unified memory)
    float *X, *Y, *Z;
    cudaMallocManaged(&X, ARRAY_BYTES);
    cudaMallocManaged(&Y, ARRAY_BYTES);
    cudaMallocManaged(&Z, ARRAY_BYTES);   // output array: never touched on the host before the kernel

    float* Z_ref = (float*)malloc(ARRAY_BYTES);   // answer key (C version)

    // initialize X and Y in host (same initialization as the SIMD specification)
    //   X[i] = sin(i * 0.001), Y[i] = cos(i * 0.002), a = 2.0
    for (int i = 0; i < ARRAY_SIZE; i++) {
        X[i] = sinf(i * 0.001f);
        Y[i] = cosf(i * 0.002f);
    }

    // C version (sanity check answer key)
    axpy_c(ARRAY_SIZE, a, X, Y, Z_ref);

    // prefetch the INPUT arrays to the GPU before calling the kernel.
    // Z is deliberately NOT prefetched (no page creation on the device side).
    int device = -1;
    cudaGetDevice(&device);
#if CUDART_VERSION >= 13000
    // CUDA 13+: location struct + flags argument
    cudaMemLocation loc;
    loc.type = cudaMemLocationTypeDevice;
    loc.id = device;
    cudaMemPrefetchAsync(X, ARRAY_BYTES, loc, 0, NULL);
    cudaMemPrefetchAsync(Y, ARRAY_BYTES, loc, 0, NULL);
#else
    // CUDA 12 and below (as in the slides)
    cudaMemPrefetchAsync(X, ARRAY_BYTES, device, NULL);
    cudaMemPrefetchAsync(Y, ARRAY_BYTES, device, NULL);
#endif
    cudaDeviceSynchronize();   // prefetch time is NOT part of the kernel time

    int numBlocks = (ARRAY_SIZE + numThreads - 1) / numThreads;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    double total_time = 0.0, min_time = 1e30, max_time = 0.0, first_time = 0.0;

    for (int run = 0; run < NUM_RUNS; run++) {
        cudaEventRecord(start);
        axpy<<<numBlocks, numThreads>>>(ARRAY_SIZE, a, X, Y, Z);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);     // wait for GPU to finish

        float ms = 0.0f;
        cudaEventElapsedTime(&ms, start, stop);
        double us = ms * 1000.0;        // microseconds
        if (run == 0) first_time = us;
        total_time += us;
        if (us < min_time) min_time = us;
        if (us > max_time) max_time = us;
    }
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) printf("CUDA error: %s\n", cudaGetErrorString(err));

    double avg_time = total_time / NUM_RUNS;
    double bw = 3.0 * (double)ARRAY_BYTES / (avg_time * 1e-6) / 1e9;   // (RB+WB)/t

    printf("\nCUDA Version (blocks = %d, threads/block = %d):\n", numBlocks, numThreads);
    printf("Kernel time (us)  avg = %.3f | min = %.3f | max = %.3f | 1st run = %.3f\n",
           avg_time, min_time, max_time, first_time);
    printf("Kernel time (ms)  avg = %.6f | min = %.6f | max = %.6f\n",
           avg_time / 1000.0, min_time / 1000.0, max_time / 1000.0);
    printf("Effective bandwidth = %.2f GB/s\n", bw);
    print_samples("CUDA Vector Z", Z, ARRAY_SIZE);

    int errors = compare(Z_ref, Z, ARRAY_SIZE);
    printf("Error Count = %d  -> %s\n", errors, errors == 0 ? "CORRECT" : "INCORRECT");

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);
    free(Z_ref);
}

int main(int argc, char** argv)
{
    float a = 2.0f;

    // Usage: ./cuda_prefetch            -> all sizes x all block sizes
    //        ./cuda_prefetch N BLOCK    -> single configuration (for nsys / ncu)
    if (argc == 3) {
        run_test(atoi(argv[1]), a, atoi(argv[2]));
        return 0;
    }

    int test_sizes[] = {
        1003,           // boundary
        1 << 20,        // 2^20
        (1 << 20) + 7,  // 2^20 + 7 (boundary)
        1 << 26,        // 2^26
        1 << 28         // 2^28
    };
    int block_sizes[] = { 256, 512, 1024 };

    for (int t = 0; t < 5; t++)
        for (int b = 0; b < 3; b++)
            run_test(test_sizes[t], a, block_sizes[b]);

    return 0;
}


Overwriting cuda_prefetch.cu


In [ ]:
# compile
!nvcc -O2 -arch=native cuda_prefetch.cu -o cuda_prefetch

### Run: output, correctness check and execution time (all sizes x all block sizes)

In [ ]:
!./cuda_prefetch


 CUDA (grid-stride + prefetch, NO page creation, NO mem advise)
 n = 1003, a = 2.00, block size = 256

CUDA Version (blocks = 4, threads/block = 256):
Kernel time (us)  avg = 49.474 | min = 6.176 | max = 1239.296 | 1st run = 1239.296
Kernel time (ms)  avg = 0.049474 | min = 0.006176 | max = 1.239296
Effective bandwidth = 0.24 GB/s
CUDA Vector Z Elements:
  First 5: 1.0000 1.0020 1.0040 1.0060 1.0080
  Last 5:  1.2683 1.2675 1.2668 1.2661 1.2653
Error Count = 0  -> CORRECT

 CUDA (grid-stride + prefetch, NO page creation, NO mem advise)
 n = 1003, a = 2.00, block size = 512

CUDA Version (blocks = 2, threads/block = 512):
Kernel time (us)  avg = 16.421 | min = 6.112 | max = 297.056 | 1st run = 297.056
Kernel time (ms)  avg = 0.016421 | min = 0.006112 | max = 0.297056
Effective bandwidth = 0.73 GB/s
CUDA Vector Z Elements:
  First 5: 1.0000 1.0020 1.0040 1.0060 1.0080
  Last 5:  1.2683 1.2675 1.2668 1.2661 1.2653
Error Count = 0  -> CORRECT

 CUDA (grid-stride + prefetch, NO page creati

### Profile with nsys (2^24 elements, block sizes 256 / 512 / 1024)

In [ ]:
for bs in (256, 512, 1024):
    print(f"==== block size {bs} ====")
    !nsys profile --stats=true --force-overwrite=true -o nsys_prefetch_{bs} ./cuda_prefetch 16777216 {bs}

==== block size 256 ====

 CUDA (grid-stride + prefetch, NO page creation, NO mem advise)
 n = 16777216, a = 2.00, block size = 256

CUDA Version (blocks = 65536, threads/block = 256):
Kernel time (us)  avg = 1149.372 | min = 825.056 | max = 10461.632 | 1st run = 10461.632
Kernel time (ms)  avg = 1.149372 | min = 0.825056 | max = 10.461632
Effective bandwidth = 175.16 GB/s
CUDA Vector Z Elements:
  First 5: 1.0000 1.0020 1.0040 1.0060 1.0080
  Last 5:  1.1895 1.1882 1.1882 1.1868 1.1868
Error Count = 0  -> CORRECT
Generating '/tmp/nsys-report-e13a.qdstrm'
[1/8] [========================100%] nsys_prefetch_256.nsys-rep
[2/8] [========================100%] nsys_prefetch_256.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_prefetch_256.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)        Name     
 --------  ---------

### Profile with ncu

In [ ]:
for bs in (256, 512, 1024):
    print(f"==== block size {bs} ====")
    !ncu --set basic ./cuda_prefetch 16777216 {bs}

Streaming output truncated to the last 5000 lines.
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                 65,536
    Registers Per Thread             register/thread              26
    Shared Memory Configuration Size           Kbyte           32.77
    Driver Shared Memory Per Block        byte/block               0
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM              40
    Stack Size                                                 1,024
    Threads                                   thread      16,777,216
    # TPCs                                                        20
    Enabled TPC IDs                                 